In [1]:
import sys
sys.path.append('/Users/yaseen/Desktop/scheduler-tasks/')

from playwright_input_labels_langchain import get_inputs
from script_generator_agent import script_generator
from parser_generator_agent import parser_generator
from pydantic_model_generator_agent import model_generator
from web_explorer_agent import run_explorer
from validator_agent import run_validator



In [2]:
#input
URL = 'https://www.justdoorsuk.com/window-order.php?product=white-window-style-1'
SAMPLE_COMMENT = "White UPVC windows with one fixed light 630 x 600mm (style 1) inc standard cill, A energy rated trickle vent and fit pack Style 1 - 150mm standard cill, white, Clear, A Triple glazed with trickle vents (1) and fit pack Update Apr 24 ce shown prices inc vat  Deduct VAT added/Use URL to access and drop down lists for all styles Note Vents qty in description/comments "


In [3]:
# web explorer agent
data = run_explorer(URL)
data

/Users/yaseen/Desktop/scheduler-tasks/myenv-shed/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'price': {'label': '£65.00',
  'tag': 'div',
  'type': '',
  'name': '',
  'id': '',
  'class_name': 'card g-brd-primary rounded-0 g-mb-15 d-none d-sm-block',
  'bbox': {'left': 85.0,
   'top': 847.1875,
   'right': 530.0,
   'bottom': 1041.28125}},
 'all_prices': [{'label': '£65.00',
   'tag': 'div',
   'type': '',
   'name': '',
   'id': '',
   'class_name': 'card g-brd-primary rounded-0 g-mb-15 d-none d-sm-block',
   'bbox': {'left': 85.0,
    'top': 847.1875,
    'right': 530.0,
    'bottom': 1041.28125}}],
 'inputs': [{'label': 'Frame Width (mm)',
   'group_label': 'Dimensions:',
   'tag': 'input',
   'type': 'text',
   'name': 'framewidth',
   'id': 'framewidth',
   'bbox': {'left': 576.0,
    'top': 540.65625,
    'right': 1179.0,
    'bottom': 574.15625},
   'is_price_relevant': True},
  {'label': 'Frame Height (mm)',
   'group_label': 'Dimensions:',
   'tag': 'input',
   'type': 'text',
   'name': 'frameheight',
   'id': 'frameheight',
   'bbox': {'left': 576.0,
    'top': 62

In [4]:
# validator agent
result = run_validator(
    labels_json_path="input_jsons/data.json",
    web_url=URL,
    output_path="results.json"
)
#result

validated_data = result.model_dump()
validated_inputs = result.model_dump()['validated_inputs']


🚀 PRICE INPUT VALIDATION SYSTEM
📄 JSON File: input_jsons/data.json
🌐 Web URL: https://www.justdoorsuk.com/window-order.php?product=white-window-style-1

[1/6] 📥 Loading JSON configuration...
✅ Loaded JSON with encoding: windows-1252
✅ Loaded 31 input definitions, 1 price candidates

[2/6] 👁️ Capturing screenshot and analysing input components...
✅ Identified 12 price-relevant components

[3a/6] 🔍 Identifying prices from screenshot (vision)...
  price_with_tax   : £65.00
  price_without_tax: not found
  base_price       : £110.00

[3b/6] 🌐 Validating price candidates against live DOM...
  1/1 candidates verified in DOM
  ✓ [0] '£65.00'  xpath='//div[contains(@class, "card g-brd-primary rounded-0 g-mb-15 d-none d-sm-block")]'  → 'Total Cost\nApril Offer Save £50 Was £115.00\nNow £65.00 inc. '

[3c/6] 🤝 Matching visual clues to DOM candidates (text LLM)...
  ✓ price_with_tax: '//div[contains(@class, "card g-brd-primary rounded-0 g-mb-15 d-none d-sm-block")]' → 'Total Cost\nApril Offer Sav

In [5]:

# extract html element (text)
def extract_elements(validated_inputs):
    elements = ""
    for elem in validated_inputs:
        elements = elements + elem['label']+"\n"+elem['outer_html']+'\n\n'
    return elements


    
# steps (List)
def get_steps(sample: dict) -> list[dict]:
    steps = []

    for elem in sample:
        tag       = elem['input_data'].get("tag", "input")
        elem_name = elem['input_data'].get("name", "")
        elem_type = elem['input_data'].get("type", "").lower()
        elem_label = elem['input_data'].get("label", "")
        xpath = elem.get('xpath')


        steps.append({"label": elem_label, "xpath": xpath, "type": elem_type, "tag": tag})

    return steps

# extract xpath
def extract_price_xpath(data: dict) -> str | None:
    """
    Extract price xpath with priority:
    1. price_without_tax
    2. price_with_tax
    3. base_price
    """
    priority = ['price_without_tax', 'price_with_tax', 'base_price']
    
    for key in priority:
        entry = data.get(key)
        if isinstance(entry, dict) and entry.get('xpath'):
            return entry['xpath']
    
    return None


In [6]:

steps = get_steps(validated_inputs)
html_elements = extract_elements(validated_inputs)
PRICE_XPATH = extract_price_xpath(validated_data['price_xpaths'])


In [2]:
# Agent1 - pydantic model generator

SAMPLE_HTML = """label name - Frame Width (mm)

<input name="framewidth" type="text" required="" class="form-control form-control-md rounded-0 g-mb-10" id="framewidth" data-msg-required="Please enter a frame width" is="dmx-input" value="" placeholder="Please enter a width between 300 - 1300" data-msg-min="Please enter a width greater than 300mm" data-msg-max="Please enter a width less than 1300mm" max="1300" data-rule-max="1300" min="300" data-rule-min="300"/>
                  
label name - Frame Height (mm)

<input name="frameheight" type="text" required="" class="form-control form-control-md rounded-0 g-mb-10" id="frameheight" data-msg-required="Please enter a frame height" is="dmx-input" value="" placeholder="Please enter a height between 300 - 2200" data-msg-min="Please enter a frame height greater than 300mm" data-msg-max="Please enter a frame height less than 2200mm" max="2200" data-rule-max="2200" min="300" data-rule-min="300"/>
                  
label name - No

<input name="cill" type="radio" required="" class="cill form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" id="cillno" is="dmx-radio" data-msg-required="Please select a cill option" value="No"/>
                   
                    
label name - 85mm Stub

<input class="cill form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" name="cill" type="radio" id="cill85" is="dmx-radio" value="85mm"/>
                    
                    
label name - Standard 150mm

<input name="cill" type="radio" class="cill form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" id="cill150" is="dmx-radio" value="150mm"/>
                    
                    
label name - 180mm

<input name="cill" type="radio" class="cill form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" id="cill180" is="dmx-radio" value="180mm"/>
                    
                    
label name - White

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="White" id="White" data-price="1.00"/>
                    
label name - Oak Both Side

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Oak Both Sides" id="Oak Both Sides" data-price="1.30"/>
                    
label name - Oak/White

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Oak/White" id="Oak/White" data-price="1.30"/>
                    
label name - Rosewood Both Sides

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Rosewood Both Sides" id="Rosewood Both Sides" data-price="1.30"/>
                    
label name - Rosewood/White

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Rosewood/White" id="Rosewood/White" data-price="1.30"/>
                    
label name - Anthracite Grev Both Sides

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Anthracite Grey Both Sides" id="Anthracite Grey Both Sides" data-price="1.30"/>
                    
label name - Anthracite Grev/White

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Anthracite Grey/White" id="Anthracite Grey/White" data-price="1.30"/>
                    
label name - Chartwell/White

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Chartwell/White" id="Chartwell/White" data-price="1.50"/>
                    
label name - Cream Both Sides

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Cream Both Sides" id="Cream Both Sides" data-price="1.30"/>
                    
label name - Cream/White

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Cream/White" id="Cream/White" data-price="1.30"/>
                    
label name - Black-Brown Both Sides

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Black-Brown Both Sides" id="Black-Brown Both Sides" data-price="1.30"/>
                    
label name - Black-Brown/White

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Black-Brown/White" id="Black-Brown/White" data-price="1.30"/>
                    
label name - Whitegrain Both Sides

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Whitegrain Both Sides" id="Whitegrain Both Sides" data-price="1.30"/>
                    
label name - Irish Oak Both Sides

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Irish Oak Both Sides" id="Irish Oak Both Sides" data-price="1.80"/>
                    
label name - Smooth Anthracite Grey/White

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Smooth Anthracite Grey/White" id="Smooth Anthracite Grey/White" data-price="1.30"/>
                    
label name - Agate Grey/White

<input name="colour" type="radio" required="" class="colour form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" data-msg-required="Please select a colour" value="Agate Grey/White" id="Agate Grey/White" data-price="1.30"/>
                    
label name - Clear

<input name="glass" type="radio" required="" class="clear form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" id="clear" onclick="" data-msg-required="Please select a glass type" value="Clear"/>
                   
                  
label name - Obscure

<input class="obscure form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" name="glass" is="dmx-radio" type="radio" id="obscure" value="Obscure"/>
                  
label name - Standard A Rated

<input name="argonglass" type="radio" required="" class="thermal form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" is="dmx-radio" id="arated" onclick="" data-msg-required="Please select an energy rating" value="A Rated"/>
                   
                  
label name - A+ Rated Energy Upgrade

<input class="aplusrated form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" name="argonglass" is="dmx-radio" type="radio" id="aplusrated" value="A+ Rated"/>
                  
label name - A++ Triple Glazed

<input class="tripleglazed form-control g-hidden-xs-up g-pos-abs g-top-0 g-left-0" name="argonglass" is="dmx-radio" type="radio" id="tripleglazed" value="A++ Triple Glazed"/>
                  
label name - Toughened Glass

<input class="toughened g-hidden-xs-up g-pos-abs g-top-0 g-left-0" name="toughenedglass" type="checkbox" id="toughened" is="dmx-checkbox" value="Toughened Glass"/>
                  
                  
label name - Laminated Glass

<input class="laminated g-hidden-xs-up g-pos-abs g-top-0 g-left-0" name="laminatedglass" type="checkbox" id="laminated" is="dmx-checkbox" value="Laminated Glass"/>
                  
                  
label name - Trickle Vents

<select class="tricklevents g-ml-10" name="tricklevents" id="tricklevents" is="dmx-select">
                <option data-price="0" value="Not Required">Not Required</option>
                <option data-price="15" value="1">1</option>
                <option data-price="30" value="2">2</option>
                </select>
                
label name - Fit Pack

<input class="fitpack g-hidden-xs-up g-pos-abs g-top-0 g-left-0" name="fitpack" id="fitpack" type="checkbox" data-price="15" is="dmx-checkbox" value="Yes"/>
                 
                  



"""


In [43]:
SAMPLE_HTML

'Frame Width (mm)\n<input name="framewidth" type="text" required="" class="form-control form-control-md rounded-0 g-mb-10" id="framewidth" data-msg-required="Please enter a frame width" is="dmx-input" value="" placeholder="Please enter a width between 300 - 1300" data-msg-min="Please enter a width greater than 300mm" data-msg-max="Please enter a width less than 1300mm" max="1300" data-rule-max="1300" min="300" data-rule-min="300">\n\nFrame Height (mm)\n<input name="frameheight" type="text" required="" class="form-control form-control-md rounded-0 g-mb-10" id="frameheight" data-msg-required="Please enter a frame height" is="dmx-input" value="" placeholder="Please enter a height between 300 - 2200" data-msg-min="Please enter a frame height greater than 300mm" data-msg-max="Please enter a frame height less than 2200mm" max="2200" data-rule-max="2200" min="300" data-rule-min="300">\n\nNo\n<input name="cill" type="radio" required="" class="cill form-control g-hidden-xs-up g-pos-abs g-top-0 

In [7]:

SAMPLE_HTML = html_elements

SAMPLE_COMMENT = "White UPVC windows with one fixed light 630 x 600mm (style 1) inc standard cill, A energy rated trickle vent and fit pack Style 1 - 150mm standard cill, white, Clear, A Triple glazed with trickle vents (1) and fit pack Update Apr 24 ce shown prices inc vat  Deduct VAT added/Use URL to access and drop down lists for all styles Note Vents qty in description/comments "

model = model_generator(SAMPLE_HTML, SAMPLE_COMMENT)
print("\n--- Generated NagivationStepsModel ---\n")
print(model)

[Agent 1] NagivationStepsModel generated and validated successfully.

--- Generated NagivationStepsModel ---

from typing import Literal, Optional
from pydantic import BaseModel, Field, field_validator

class NagivationStepsModel(BaseModel):
    model_config = {"populate_by_name": True}

    Frame_Width_mm: Optional[str] = Field(None, alias="Frame Width (mm)")
    Frame_Height_mm: Optional[str] = Field(None, alias="Frame Height (mm)")
    No: Optional[bool] = Field(None, alias="No")
    mm85_Stub: Optional[bool] = Field(None, alias="85mm Stub")
    Standard_mm150: Optional[bool] = Field(None, alias="Standard 150mm")
    mm180: Optional[bool] = Field(None, alias="180mm")
    White: Optional[bool] = Field(None, alias="White")
    Oak_Both_Sides: Optional[bool] = Field(None, alias="Oak Both Sides")
    Oak_White: Optional[bool] = Field(None, alias="Oak/White")
    Rosewood_Both_Sides: Optional[bool] = Field(None, alias="Rosewood Both Sides")
    Rosewood_White: Optional[bool] = Field(None

In [ ]:


from typing import Literal, Optional
from pydantic import BaseModel, Field, field_validator

class NagivationStepsModel(BaseModel):
    model_config = {"populate_by_name": True}
    Frame_Width_mm: Optional[str] = Field(None, alias="Frame Width (mm)")
    Frame_Height_mm: Optional[str] = Field(None, alias="Frame Height (mm)")
    No: Optional[bool] = Field(None, alias="No")
    mm85_Stub: Optional[bool] = Field(None, alias="85mm Stub")
    Standard_150mm: Optional[bool] = Field(None, alias="Standard 150mm")
    mm180: Optional[bool] = Field(None, alias="180mm")
    White: Optional[bool] = Field(None, alias="White")
    Oak_Both_Side: Optional[bool] = Field(None, alias="Oak Both Side")
    Oak_White: Optional[bool] = Field(None, alias="Oak/White")
    Rosewood_Both_Sides: Optional[bool] = Field(None, alias="Rosewood Both Sides")
    Rosewood_White: Optional[bool] = Field(None, alias="Rosewood/White")
    Anthracite_Grey_Both_Sides: Optional[bool] = Field(None, alias="Anthracite Grev Both Sides")
    Anthracite_Grey_White: Optional[bool] = Field(None, alias="Anthracite Grev/White")
    Chartwell_White: Optional[bool] = Field(None, alias="Chartwell/White")
    Cream_Both_Sides: Optional[bool] = Field(None, alias="Cream Both Sides")
    Cream_White: Optional[bool] = Field(None, alias="Cream/White")
    Black_Brown_Both_Sides: Optional[bool] = Field(None, alias="Black-Brown Both Sides")
    Black_Brown_White: Optional[bool] = Field(None, alias="Black-Brown/White")
    Whitegrain_Both_Sides: Optional[bool] = Field(None, alias="Whitegrain Both Sides")
    Irish_Oak_Both_Sides: Optional[bool] = Field(None, alias="Irish Oak Both Sides")
    Smooth_Anthracite_Grey_White: Optional[bool] = Field(None, alias="Smooth Anthracite Grey/White")
    Agate_Grey_White: Optional[bool] = Field(None, alias="Agate Grey/White")
    Clear: Optional[bool] = Field(None, alias="Clear")
    Obscure: Optional[bool] = Field(None, alias="Obscure")
    Standard_A_Rated: Optional[bool] = Field(None, alias="Standard A Rated")
    A_Plus_Rated_Energy_Upgrade: Optional[bool] = Field(None, alias="A+ Rated Energy Upgrade")
    A_Plus_Plus_Triple_Glazed: Optional[bool] = Field(None, alias="A++ Triple Glazed")
    Toughened_Glass: Optional[bool] = Field(None, alias="Toughened Glass")
    Laminated_Glass: Optional[bool] = Field(None, alias="Laminated Glass")
    Trickle_Vents: str = Field("", alias="Trickle Vents")
    Fit_Pack: Optional[bool] = Field(None, alias="Fit Pack")

    @field_validator("Trickle_Vents", mode="before")
    def validate_trickle_vents(cls, v):
        map_ = {
            "Not Required": "Not Required",
            "1": "1",
            "2": "2"
        }
        return map_.get(str(v), str(v))

    

In [8]:
SAMPLE_MODEL = model

In [ ]:
# Test directly
m = NagivationStepsModel(
    No=True,
    mm85_Stub=None,
    Standard_150mm=None,
    mm180=None,
)
print(m.model_dump(by_alias=True))
# Does 'No' show True or None?

In [9]:

SAMPLE_MODEL = model
SAMPLE_COMMENT = "White UPVC windows with one fixed light 630 x 600mm (style 1) inc standard cill, A energy rated trickle vent and fit pack Style 1 - 150mm standard cill, white, Clear, A Triple glazed with trickle vents (1) and fit pack Update Apr 24 ce shown prices inc vat  Deduct VAT added/Use URL to access and drop down lists for all styles Note Vents qty in description/comments "


# Agent2
parser = parser_generator(SAMPLE_MODEL, SAMPLE_COMMENT)

[Agent 2] parse_comment() generated and validated successfully.


In [11]:
print(parser)

def parse_comment(comment: str) -> NagivationStepsModel:
    c = comment.lower()
    # --- Dimensions ---
    dim = re.search(r'(\d+)\s*[x×/-]\s*(\d+)', c)
    if dim:
        width, height = dim.group(1), dim.group(2)
    else:
        return parse_order_from_comment(comment)
    # --- Radio group ---
    if '85mm stub' in c:
        selected_cill = '85mm Stub'
    elif 'standard 150mm' in c:
        selected_cill = 'Standard 150mm'
    elif '180mm' in c:
        selected_cill = '180mm'
    else:
        selected_cill = None
    # --- Colour group ---
    colours = ['white', 'oak both sides', 'oak/white', 'rosewood both sides', 'rosewood/white',
               'anthracite grey both sides', 'anthracite grey/white', 'chartwell/white',
               'cream both sides', 'cream/white', 'black-brown both sides', 'black-brown/white',
               'whitegrain both sides', 'irish oak both sides', 'smooth anthracite grey/white',
               'agate grey/white']
    colour_fields = ['White'

In [16]:
# parser function
PARSER = parser

# web-action steps
STEPS = """[
    {'label': 'Frame Width (mm)',  'xpath': "//input[@id='framewidth']",              'type': 'text',     'tag': 'input'},
    {'label': 'Frame Height (mm)', 'xpath': "//input[@id='frameheight']",             'type': 'text',     'tag': 'input'},
    {'label': 'No',                'xpath': "//input[@id='cillno']",                  'type': 'radio',    'tag': 'input'},
    {'label': '85mm Stub',         'xpath': "//input[@id='cill85']",                  'type': 'radio',    'tag': 'input'},
    {'label': 'Standard 150mm',    'xpath': "//input[@id='cill150']",                 'type': 'radio',    'tag': 'input'},
    {'label': '180mm',             'xpath': "//input[@id='cill180']",                 'type': 'radio',    'tag': 'input'},
    {'label': 'White',             'xpath': "//input[@id='White']",                   'type': 'radio',    'tag': 'input'},
    {'label': 'Oak Both Sides',    'xpath': "//input[@id='Oak Both Sides']",           'type': 'radio',    'tag': 'input'},
    {'label': 'Oak/White',         'xpath': "//input[@id='Oak/White']",               'type': 'radio',    'tag': 'input'},
    {'label': 'Rosewood Both Sides',       'xpath': "//input[@id='Rosewood Both Sides']",       'type': 'radio', 'tag': 'input'},
    {'label': 'Rosewood/White',            'xpath': "//input[@id='Rosewood/White']",            'type': 'radio', 'tag': 'input'},
    {'label': 'Anthracite Grey Both Sides','xpath': "//input[@id='Anthracite Grey Both Sides']",'type': 'radio', 'tag': 'input'},
    {'label': 'Anthracite Grey/White',     'xpath': "//input[@id='Anthracite Grey/White']",     'type': 'radio', 'tag': 'input'},
    {'label': 'Chartwell/White',           'xpath': "//input[@id='Chartwell/White']",           'type': 'radio', 'tag': 'input'},
    {'label': 'Cream Both Sides',          'xpath': "//input[@id='Cream Both Sides']",          'type': 'radio', 'tag': 'input'},
    {'label': 'Cream/White',               'xpath': "//input[@id='Cream/White']",               'type': 'radio', 'tag': 'input'},
    {'label': 'Black-Brown Both Sides',    'xpath': "//input[@id='Black-Brown Both Sides']",    'type': 'radio', 'tag': 'input'},
    {'label': 'Black-Brown/White',         'xpath': "//input[@id='Black-Brown/White']",         'type': 'radio', 'tag': 'input'},
    {'label': 'Whitegrain Both Sides',     'xpath': "//input[@id='Whitegrain Both Sides']",     'type': 'radio', 'tag': 'input'},
    {'label': 'Irish Oak Both Sides',      'xpath': "//input[@id='Irish Oak Both Sides']",      'type': 'radio', 'tag': 'input'},
    {'label': 'Smooth Anthracite Grey/White', 'xpath': "//input[@id='Smooth Anthracite Grey/White']", 'type': 'radio', 'tag': 'input'},
    {'label': 'Agate Grey/White',          'xpath': "//input[@id='Agate Grey/White']",          'type': 'radio', 'tag': 'input'},
    {'label': 'Clear',             'xpath': "//input[@id='clear']",                   'type': 'radio',    'tag': 'input'},
    {'label': 'Obscure',           'xpath': "//input[@id='obscure']",                 'type': 'radio',    'tag': 'input'},
    {'label': 'Standard A Rated',  'xpath': "//input[@id='arated']",                  'type': 'radio',    'tag': 'input'},
    {'label': 'A++ Triple Glazed', 'xpath': "//input[@id='tripleglazed']",            'type': 'radio',    'tag': 'input'},
    {'label': 'Toughened Glass',   'xpath': "//input[@id='toughened']",               'type': 'checkbox', 'tag': 'input'},
    {'label': 'Laminated Glass',   'xpath': "//input[@id='laminated']",               'type': 'checkbox', 'tag': 'input'},
    {'label': 'Trickle Vents',     'xpath': "//select[@id='tricklevents']",           'type': '',         'tag': 'select'},
    {'label': 'Fit Pack',          'xpath': "//input[@id='fitpack']",                 'type': 'checkbox', 'tag': 'input'},
]
"""










In [15]:
STEPS = str(steps)
STEPS

'[{\'label\': \'Frame Width (mm)\', \'xpath\': \'//*[@id="framewidth"]\', \'type\': \'text\', \'tag\': \'input\'}, {\'label\': \'Frame Height (mm)\', \'xpath\': \'//*[@id="frameheight"]\', \'type\': \'text\', \'tag\': \'input\'}, {\'label\': \'No\', \'xpath\': \'//*[@id="cillno"]\', \'type\': \'radio\', \'tag\': \'input\'}, {\'label\': \'85mm Stub\', \'xpath\': \'//*[@id="cill85"]\', \'type\': \'radio\', \'tag\': \'input\'}, {\'label\': \'Standard 150mm\', \'xpath\': \'//*[@id="cill150"]\', \'type\': \'radio\', \'tag\': \'input\'}, {\'label\': \'180mm\', \'xpath\': \'//*[@id="cill180"]\', \'type\': \'radio\', \'tag\': \'input\'}, {\'label\': \'White\', \'xpath\': \'//*[@id="White"]\', \'type\': \'radio\', \'tag\': \'input\'}, {\'label\': \'Oak Both Sides\', \'xpath\': \'//*[@id="Oak Both Sides"]\', \'type\': \'radio\', \'tag\': \'input\'}, {\'label\': \'Oak/White\', \'xpath\': \'//*[@id="Oak/White"]\', \'type\': \'radio\', \'tag\': \'input\'}, {\'label\': \'Rosewood Both Sides\', \'xpa

In [18]:
STEPS = str(steps)

# Generate script - Agent3
script_generator(
        comment_parser_function=PARSER,
        steps=STEPS,
        price_xpath=PRICE_XPATH,
        pydantic_model = SAMPLE_MODEL,
        output_path="scraper_output3.py",
)

[Agent 3] Script written to: scraper_output3.py


'import re\nimport time\nimport random\nfrom playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError, expect\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom langchain_openai import ChatOpenAI\nfrom pydantic import BaseModel, Field\nfrom typing import Literal, Optional\nimport dotenv\n\n\ndotenv.load_dotenv()\n\n\nSYSTEM_PROMPT = """\nYou are a form-filling assistant.\n\nYou will be given:\n1. A Pydantic model definition representing a web form\n2. A customer comment describing what they want\n\nYour job is to extract values from the comment and return ONLY a filled Pydantic object constructor call — nothing else. No explanation, no code block, no markdown.\n\nRules:\n- Return only the Pydantic constructor call e.g. ModelName(field=value, ...)\n- Only include fields that have a value extracted from the comment\n- For numerical values take numerical values from the comment and convert to string (e.g. "630mm" -> "630")\n- For radio button fields (Op